In [1]:
%load_ext autoreload
%autoreload 2

In [1]:
import sys
sys.path = ['/data/mhoffert/tools/ete/'] + sys.path[:-1]
from ete3 import Tree
from ete3.treeview import faces, AttrFace, TextFace, TreeStyle, NodeStyle, CircleFace, RectFace

# python libraries
import pickle
import itertools
import random
import os


# data
import pandas as pd
import numpy as np



# plotting
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.collections import PolyCollection
import matplotlib
import matplotlib.cm as cm
import matplotlib.patches as patches
from matplotlib.colors import to_hex

# additional plotting code
import sys
import time

# displays
from IPython.display import display, clear_output

# for visualizing trees
os.environ['QT_QPA_PLATFORM']='offscreen'

## Load GTDB data

In [2]:
# read in gtdb metadata
md = pd.read_csv('/data/mhoffert/genomes/GTDB_r207/bac120_metadata_r207.tsv', sep='\t')

# filter to representatives
md = md[md.gtdb_representative.eq('t')].copy(deep=True)
md['Phylum'] = md.gtdb_taxonomy.apply(lambda x: x.split(';')[1].split('__')[-1])
md['Class'] = md.gtdb_taxonomy.apply(lambda x: x.split(';')[2].split('__')[-1])
md['Order'] = md.gtdb_taxonomy.apply(lambda x: x.split(';')[3].split('__')[-1])
md['Family'] = md.gtdb_taxonomy.apply(lambda x: x.split(';')[4].split('__')[-1])
md['Genus'] = md.gtdb_taxonomy.apply(lambda x: x.split(';')[5].split('__')[-1])
md['Species'] = md.gtdb_taxonomy.apply(lambda x: x.split(';')[6].split('__')[-1])

# subset to phyla with at least 100 representatives
phylum_counts = md.groupby('Phylum').count()
top_phyla = phylum_counts[phylum_counts['accession'] > 100].index.unique()
md_top_phyla = md[md['Phylum'].isin(top_phyla)].copy(deep=True)

Columns (61,65,74,82,83,85) have mixed types. Specify dtype option on import or set low_memory=False.


In [3]:
# additional columns
phylum_counts = md_top_phyla.groupby('Phylum').count()['accession']
class_counts = md_top_phyla.groupby('Class').count()['accession']

# parsing to make plot of isolate vs. environmental
md_top_phyla['ncbi_genome_category_grouped'] = md_top_phyla['ncbi_genome_category'].apply(lambda x: 'Isolate' if x=='none' else 'MAG/SAG/environmental')

In [4]:
md_top_phyla['accession_reformat'] = md_top_phyla['accession'].apply(lambda x: x.replace('.', '_'))

In [5]:
# get a list of the most abundant phyla
most_abundant_phyla = md_top_phyla.groupby(['Phylum']).count().sort_values('accession', ascending=False).head(10)
print(most_abundant_phyla['accession'])

Phylum
Proteobacteria       17350
Bacteroidota          8588
Firmicutes_A          8243
Actinobacteriota      7328
Firmicutes            4216
Patescibacteria       2485
Chloroflexota         1387
Cyanobacteria         1372
Verrucomicrobiota     1325
Planctomycetota       1071
Name: accession, dtype: int64


In [6]:
md['ncbi_isolation_source'].value_counts()

ncbi_isolation_source
none                                      56895
soil                                        306
biological product [ENVO:02000043]          171
derived from human gut metagenome           156
not provided; submitted under MIGS 2.1      136
                                          ...  
a paper mill waste water polluted pond        1
soil sediment                                 1
grasslands                                    1
soil of a rice field                          1
host s whole body                             1
Name: count, Length: 2026, dtype: int64

### Loading in the tree

In [7]:
gtdb_full = Tree('/data/mhoffert/genomes/GTDB_r207/tree/bac120_r207.tree', format=1, quoted_node_names=True)

### Reformatting tree

In [8]:
# rename internal nodes
names_map = []
i = 0
for node in gtdb_full.traverse(strategy='levelorder'):
    if i % 500 == 0:
        display(i)
        clear_output(wait=True)
    if not node.is_leaf():
        new_node_name = f'c{i:06}'
        names_map.append((new_node_name, node.name))
        node.name = new_node_name
    else:
        names_map.append((node.name.replace('.', '_'), node.name))
        node.name = node.name.replace('.', '_')
    i += 1

124500

In [9]:
# make a series
names_map = pd.Series(dict(names_map))

In [10]:
phylum_counts.sort_values(ascending=False)

Phylum
Proteobacteria        17350
Bacteroidota           8588
Firmicutes_A           8243
Actinobacteriota       7328
Firmicutes             4216
Patescibacteria        2485
Chloroflexota          1387
Cyanobacteria          1372
Verrucomicrobiota      1325
Planctomycetota        1071
Desulfobacterota        939
Acidobacteriota         873
Spirochaetota           695
Campylobacterota        550
Firmicutes_C            395
Myxococcota             393
Firmicutes_B            323
Nitrospirota            314
Omnitrophota            282
Gemmatimonadota         237
Desulfobacterota_I      236
Bdellovibrionota        224
Elusimicrobiota         189
Marinisomatota          186
Chlamydiota             171
Deinococcota            144
Armatimonadota          139
Synergistota            132
Firmicutes_G            131
Fibrobacterota          120
Firmicutes_D            104
Name: accession, dtype: int64

In [14]:
# construct collapsing method
def collapseTree(tree, func, **collapse_args): 
    '''
    Master TreeNode collapsing function
    Collapses a TreeNode using specified leaf finding function
    '''
    # collapse tree using function
    t = Tree(tree.write(is_leaf_fn=lambda n: func(n, **collapse_args), format=3),format=3) 
    t.name = tree.name
    return t

def collapse_to_phylum(node, nm):
    '''
    collapses to phylum root nodes
    '''
    if node.is_leaf():
        return True
    elif node.up is None:
        return False
    else:
        return 'p__' in nm.loc[node.name]
    
def collapse_count(node):
    '''
    collapses to phylum root nodes
    '''
    if node.is_leaf():
        return True
    elif node.up is None:
        return False
    else:
        return len(node.get_leaves()) < 400

In [12]:
def layout_collapsed_phylum(node, clades, inc, mrc, rc):
    '''
    Layout function for tree generated by collapse_to_phylum
    '''
    # print(node.name)
    internal_ns = NodeStyle()
    if node.name in clades:
        print('yes')
        internal_ns['size'] = 10
        internal_ns['fgcolor'] = 'red'
    else:
        internal_ns['size'] = 0
        
    if node.name in inc.keys():
        internal_ns['vt_line_color'] = inc[node.name]
        internal_ns['hz_line_color'] = inc[node.name]
    
    if node.is_leaf():
        
        ringcolor = inc[node.name]
        
        R1 = RectFace(width=4, height=4, fgcolor=ringcolor, bgcolor=ringcolor)
        faces.add_face_to_node(R1, node, 0, position='aligned')
        
        Rs = RectFace(width=2, height=4, fgcolor='white', bgcolor='white')
        faces.add_face_to_node(Rs, node, 1, position='aligned')
        
        ringcolor = mrc[node.name]
        
        R1 = RectFace(width=12, height=4, fgcolor=ringcolor, bgcolor=ringcolor)
        faces.add_face_to_node(R1, node, 2, position='aligned')
        
        Rs = RectFace(width=2, height=4, fgcolor='white', bgcolor='white')
        faces.add_face_to_node(Rs, node, 3, position='aligned')
        
        ringcolor = rc[node.name]
        
        R2 = RectFace(width=12, height=4, fgcolor=ringcolor, bgcolor=ringcolor)
        faces.add_face_to_node(R2, node, 4, position='aligned')
        
        Rs = RectFace(width=2, height=4, fgcolor='white', bgcolor='white')
        faces.add_face_to_node(Rs, node, 5, position='aligned')
#         new_name = names_map.loc[node.name].split(':')[-1].split(';')[0].lstrip()
        
#         faces.add_face_to_node(TextFace(f" {new_name}", fsize=6), node, column=0)
        
#         num_leaves = len(gtdb_full.search_nodes(name=node.name)[0].get_leaves())
#         faces.add_face_to_node(TextFace(f"{num_leaves}", fsize=6), node, column=1, position='aligned')
    
#         face_color='k'
#         if num_leaves >= 1900:
            
#             face_color='red'
            
#         # face_color = node_size_cmap(len(node.get_leaves()) / 2000)
#         R1 = RectFace(width=num_leaves / 100,
#                       height=3, 
#               fgcolor=face_color,
#               bgcolor=face_color)
#         faces.add_face_to_node(R1, node, 2, position='aligned')

        
#         internal_ns['fgcolor'] = face_color
#         internal_ns['size'] = 5

    node.img_style = internal_ns

def phylum_collapse_ts(**args):
    ts = TreeStyle()
    ts.mode = 'c'
    ts.layout_fn = lambda n: layout_collapsed_phylum(n, **args)
    # ts.scale = 400
    ts.show_leaf_name = False
    ts.draw_guiding_lines = False
    
    return ts

In [15]:
collapsed = collapseTree(gtdb_full, collapse_to_phylum, nm=names_map)

collapsed.describe()

Number of leaf nodes:	169
Total number of nodes:	337
Rooted:	Yes
Most distant node:	c009809
Max. distance:	1.364140
